# Read-only Iceberg HadoopCatalog exploration

This notebook inspects a persistent HadoopCatalog created by the Java loader. It deliberately contains no write, maintenance, or catalog-mutation commands.

Edit the next configuration cell to explore another catalog, namespace, table, or UTC range.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os
import sys

# Kernels may start in the notebook directory rather than the checkout root.
configured_root = os.environ.get('GCE_HADOOP_CATALOG_REPOSITORY')
candidates = ([Path(configured_root).expanduser()] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
repository_root = next(
    (path.resolve() for path in candidates if (path / 'src' / 'gce_hadoop_catalog').is_dir()),
    None,
)
if repository_root is None:
    raise RuntimeError(
        'Cannot find the checkout. Start Jupyter from the repository or set '
        'GCE_HADOOP_CATALOG_REPOSITORY=/path/to/gcp_alpaca_datalake.'
    )
sys.path.insert(0, str(repository_root / 'src'))

from gce_hadoop_catalog.spark_catalog import create_spark_session, settings_from_environment

## Notebook configuration

Edit the values in the next cell. Namespace, table, range, and limit changes need only that cell and the later query cells to be rerun. Warehouse, catalog, GCS mode, or connector changes require **Restart Kernel**, then rerun from the imports cell.

In [ ]:
# These initial values preserve the current launch-time defaults. Edit them directly.
defaults = settings_from_environment(repository_root)

# Spark-session settings: after changing one, restart the kernel before recreating Spark.
CATALOG_NAME = defaults.catalog_name
WAREHOUSE = defaults.warehouse
ENABLE_GCS = defaults.gcs_enabled
GCS_CONNECTOR_JAR = str(defaults.gcs_connector_jar)

# Query target: edit and rerun this cell plus the discovery/query cells.
NAMESPACE = defaults.namespace
TABLE = defaults.table

# Bounded UTC exploration query.
START_UTC = '2026-01-01T00:00:00Z'
END_UTC = '2026-01-03T00:00:00Z'
RESULT_LIMIT = 100

settings = replace(
    defaults,
    catalog_name=CATALOG_NAME,
    warehouse=WAREHOUSE,
    gcs_enabled=ENABLE_GCS,
    gcs_connector_jar=Path(GCS_CONNECTOR_JAR),
    namespace=NAMESPACE,
    table=TABLE,
)
if settings.warehouse.startswith('gs://') and not settings.gcs_enabled:
    raise ValueError('Set ENABLE_GCS = True before opening a gs:// warehouse.')
if RESULT_LIMIT <= 0:
    raise ValueError('RESULT_LIMIT must be positive.')

print({
    'catalog': settings.catalog_name,
    'warehouse': settings.warehouse,
    'namespace': settings.namespace,
    'table': settings.table,
    'gcs_enabled': settings.gcs_enabled,
    'start_utc': START_UTC,
    'end_utc': END_UTC,
    'result_limit': RESULT_LIMIT,
})

In [ ]:
spark = create_spark_session(settings)
spark.conf.get('spark.sql.session.timeZone')

## Inspect the configured namespace

HadoopCatalog does not reliably list a completely empty warehouse root, so this notebook inspects the configured namespace directly.

In [ ]:
namespace = f'{settings.catalog_name}.{settings.namespace}'
try:
    spark.sql(f'SHOW TABLES IN {namespace}').show(truncate=False)
except Exception as error:
    raise RuntimeError(
        f'Namespace {namespace!r} is absent from {settings.warehouse!r}. '
        'For persistent local data, run: '
        'uv run python scripts/run_local_stack.py --source synthetic --runtime-dir .local-notebook; '
        'then restart Jupyter with ICEBERG_WAREHOUSE="$PWD/.local-notebook/warehouse".'
    ) from error

In [ ]:
table = settings.table_identifier
spark.sql(f'DESCRIBE TABLE {table}').show(truncate=False)
spark.sql(f'SELECT committed_at, snapshot_id, operation, summary FROM {table}.snapshots ORDER BY committed_at DESC').show(truncate=False)

## Bounded bar query

The table retains Alpaca's case-sensitive wire contract: `T` is the event type and `t` is the UTC RFC 3339 timestamp. Spark is therefore configured as case-sensitive.

In [ ]:
bars = spark.sql(
    f"""
    SELECT S AS symbol, t, o, h, l, c, v, n, vw, ingested_at, payload_hash
    FROM {table}
    WHERE t >= '{START_UTC}' AND t < '{END_UTC}'
    ORDER BY t, S
    LIMIT {RESULT_LIMIT}
    """
)
bars.show(truncate=False)

In [ ]:
spark.sql(
    f"EXPLAIN FORMATTED SELECT * FROM {table} WHERE t >= '{START_UTC}' AND t < '{END_UTC}'"
).show(truncate=False)